# Notebook 9 -- Three-Angle Evaluation on ASDiv
## SLM-to-SLM Guided Reasoning Pipeline

**Why ASDiv?**

ASDiv (Academic Subtypes Diversity) tags every problem with its operation type:
Addition, Subtraction, Multiplication, Division, and multi-step combinations.
This lets us break results down by category and ask:
*"Does guidance help more on division than addition?"*
That is a much richer finding than a single accuracy number across all problems.

| Property | SVAMP | ASDiv |
|---|---|---|
| HuggingFace ID | `ChilleD/SVAMP` | `EleutherAI/asdiv` |
| Size | 1,000 | ~2,300 |
| Format | Open numeric | Open numeric |
| Key field | None | `solution_type` -- operation category |
| Difficulty | Robustness-focused | Wider operation variety |

**Key addition vs SVAMP notebook**
- Per operation-type accuracy breakdown (Addition / Subtraction / Multiplication / Division / multi-step)
- Guidance lift measured separately per operation type
- Vote consistency broken down per operation type
- This enables the paper claim: "guidance helps most on X-type problems"

**Pipeline (identical structure)**
```
Question --> Fine-tuned Qwen 3B (Guide) --> Plan --> Qwen 1.5B (Solver) x5 --> Vote --> Answer
Baseline: Question -----------------------------------------> Qwen 1.5B x5 --> Vote
```


In [1]:
# CELL 1 -- Install (uncomment on first run)
# !pip install -q transformers==4.44.0
# !pip install -q peft==0.12.0
# !pip install -q accelerate==0.33.0
# !pip install -q datasets==2.20.0
# !pip install -q huggingface_hub
print("Done.")


Done.


In [ ]:
# CELL 2 -- HuggingFace login
from huggingface_hub import login

login("")
print("HuggingFace login done")


HuggingFace login done


In [3]:
# CELL 3 -- Imports + GPU
import os, json, re, glob, random, time
import torch
import numpy as np
from collections import Counter, defaultdict
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from tqdm.notebook import tqdm

OUTPUT_DIR = "/kaggle/working/asdiv_eval"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Output  : {OUTPUT_DIR}")


PyTorch : 2.10.0+cu128
GPU     : Tesla T4
VRAM    : 15.6 GB
Output  : /kaggle/working/asdiv_eval


In [4]:
# CELL 4 -- Configuration
CONFIG = {
    # Models
    "guide_base"          : "Qwen/Qwen2.5-3B-Instruct",
    "response_model"      : "Qwen/Qwen2.5-1.5B-Instruct",

    # Dataset
    "dataset_name"        : "EleutherAI/asdiv",
    "dataset_split"       : "validation",   # ASDiv has train/validation splits
    "max_eval_samples"    : 300,            # validation has ~2300; increase for full run
    "random_seed"         : 42,             # FIXED -- same seed for BOTH conditions

    # Ensemble
    "n_votes"             : 5,
    "vote_temperature"    : 0.7,
    "guide_temperature"   : 0.1,
    "refiner_temperature" : 0.3,
    "max_new_tokens"      : 350,

    # Compute cost (billions of parameters)
    "guide_params_B"      : 3.0,
    "solver_params_B"     : 1.5,

    # Paths
    "results_file"        : f"{OUTPUT_DIR}/results.jsonl",
    "report_file"         : f"{OUTPUT_DIR}/eval_report.json",
    "angle1_file"         : f"{OUTPUT_DIR}/angle1_compute_efficiency.json",
    "angle2_file"         : f"{OUTPUT_DIR}/angle2_vote_consistency.json",
    "angle3_file"         : f"{OUTPUT_DIR}/angle3_confidence_calibration.json",
    "angle4_file"         : f"{OUTPUT_DIR}/angle4_by_operation_type.json",
    "checkpoint_file"     : f"{OUTPUT_DIR}/checkpoint.json",
    "save_every"          : 25,
}

print("Config ready:")
for k, v in CONFIG.items():
    print(f"  {k:<24}: {v}")


Config ready:
  guide_base              : Qwen/Qwen2.5-3B-Instruct
  response_model          : Qwen/Qwen2.5-1.5B-Instruct
  dataset_name            : EleutherAI/asdiv
  dataset_split           : validation
  max_eval_samples        : 300
  random_seed             : 42
  n_votes                 : 5
  vote_temperature        : 0.7
  guide_temperature       : 0.1
  refiner_temperature     : 0.3
  max_new_tokens          : 350
  guide_params_B          : 3.0
  solver_params_B         : 1.5
  results_file            : /kaggle/working/asdiv_eval/results.jsonl
  report_file             : /kaggle/working/asdiv_eval/eval_report.json
  angle1_file             : /kaggle/working/asdiv_eval/angle1_compute_efficiency.json
  angle2_file             : /kaggle/working/asdiv_eval/angle2_vote_consistency.json
  angle3_file             : /kaggle/working/asdiv_eval/angle3_confidence_calibration.json
  angle4_file             : /kaggle/working/asdiv_eval/angle4_by_operation_type.json
  checkpoint_file      

In [5]:
# CELL 5 -- Load ASDiv dataset
# ASDiv fields: body, question, solution_type, answer, formula
#
# solution_type examples:
#   "Addition", "Subtraction", "Multiplication", "Division",
#   "Common-Division", "Comparison", "Sum", "Difference" etc.
#
# We combine body + question into a single question string,
# extract the numeric answer, and map solution_type to a
# broad 5-category label used for the per-type analysis.

print("Loading ASDiv from HuggingFace...")
raw_ds = load_dataset(CONFIG["dataset_name"])

print(f"Splits   : {list(raw_ds.keys())}")
print(f"Features : {list(raw_ds[CONFIG['dataset_split']].features.keys())}")
print(f"Split size: {len(raw_ds[CONFIG['dataset_split']])}")

ex = raw_ds[CONFIG["dataset_split"]][0]
print(f"\nExample record:")
for k, v in ex.items():
    print(f"  {k}: {v}")


# Map the fine-grained solution_type into 5 broad categories
OP_MAP = {
    "addition"         : "Addition",
    "sum"              : "Addition",
    "subtraction"      : "Subtraction",
    "difference"       : "Subtraction",
    "comparison"       : "Subtraction",
    "multiplication"   : "Multiplication",
    "division"         : "Division",
    "common-division"  : "Division",
    "floor-division"   : "Division",
}

def broad_op(solution_type):
    """Map fine-grained solution_type to one of 5 broad categories."""
    st = str(solution_type).lower().strip()
    for key, val in OP_MAP.items():
        if key in st:
            return val
    # multi-step or unknown
    return "Multi-step"


def normalise_asdiv(item):
    """Convert ASDiv record to {question, answer, op_type} pipeline format."""
    q = item["body"].strip().rstrip(".") + " " + item["question"].strip()

    # Answer may be "10 cookies" or "10" -- keep only the leading numeric part
    raw_ans = str(item["answer"]).strip().replace(",", "")
    m = re.match(r"(-?[\d\.]+)", raw_ans)
    ans_str = m.group(1) if m else raw_ans

    # Normalise float: 5.0 -> "5"
    try:
        f = float(ans_str)
        ans_str = str(int(f)) if f == int(f) else str(round(f, 4))
    except Exception:
        pass

    op_type      = broad_op(item.get("solution_type", ""))
    solution_type = str(item.get("solution_type", ""))

    return {
        "question"      : q,
        "answer"        : ans_str,
        "op_type"       : op_type,
        "solution_type" : solution_type,
    }


all_data = [normalise_asdiv(x) for x in raw_ds[CONFIG["dataset_split"]]]

# Show operation type distribution before sampling
op_counts = Counter(d["op_type"] for d in all_data)
print(f"\nOperation type distribution ({len(all_data)} total):")
for op, cnt in sorted(op_counts.items(), key=lambda x: -x[1]):
    print(f"  {op:<20}: {cnt}")

# CRITICAL: fix seed ONCE here, before any sampling
random.seed(CONFIG["random_seed"])
if CONFIG["max_eval_samples"] < len(all_data):
    test_data = random.sample(all_data, CONFIG["max_eval_samples"])
    print(f"\nSampled {len(test_data)} questions (seed={CONFIG['random_seed']})")
else:
    test_data = all_data
    print(f"\nUsing all {len(test_data)} questions")

# Show op distribution in sampled set
sampled_op = Counter(d["op_type"] for d in test_data)
print(f"\nSampled operation distribution:")
for op, cnt in sorted(sampled_op.items(), key=lambda x: -x[1]):
    print(f"  {op:<20}: {cnt}")

print(f"\nFirst Q : {test_data[0]['question'][:80]}...")
print(f"First A : {test_data[0]['answer']}  |  op: {test_data[0]['op_type']}")
print("ASDiv loaded")


Loading ASDiv from HuggingFace...


README.md:   0%|          | 0.00/494 [00:00<?, ?B/s]

asdiv/validation-00000-of-00001.parquet:   0%|          | 0.00/267k [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/2305 [00:00<?, ? examples/s]

Splits   : ['validation']
Features : ['body', 'question', 'solution_type', 'answer', 'formula']
Split size: 2305

Example record:
  body: Seven red apples and two green apples are in the basket.
  question: How many apples are in the basket?
  solution_type: Addition
  answer: 9 (apples)
  formula: 7+2=9

Operation type distribution (2305 total):
  Multi-step          : 723
  Subtraction         : 568
  Addition            : 446
  Division            : 308
  Multiplication      : 260

Sampled 300 questions (seed=42)

Sampled operation distribution:
  Multi-step          : 89
  Subtraction         : 67
  Addition            : 61
  Multiplication      : 43
  Division            : 40

First Q : He also has a section filled with short story booklets. If each booklet has 9 pa...
First A : 441  |  op: Multiplication
ASDiv loaded


In [6]:
# CELL 6 -- Answer extraction (same as SVAMP -- numeric answers)
# ASDiv answers are integers or simple floats.
# Answers sometimes include units like "10 cookies" -- we strip those in Cell 5.

def normalise_num(s):
    """Canonical numeric string. 5.0 -> '5', 3.14 -> '3.14'."""
    s = s.replace(",", "").strip()
    try:
        f = float(s)
        return str(int(f)) if f == int(f) else str(round(f, 4))
    except ValueError:
        return s


def extract_gt_answer(answer_str):
    """ASDiv GT is already cleaned in normalise_asdiv -- just normalise."""
    return normalise_num(str(answer_str))


def extract_pred_answer(text):
    """
    Multi-pattern extractor. Returns empty string on failure.
    Never falls back to a random number from the text.

    Priority:
      1. #### N          -- standard format we request
      2. \\boxed{N}      -- Qwen's preferred LaTeX style
      3. 'the answer is' -- common phrasing
      4. '= N' at end of line
      5. **N** at end    -- bold markdown
      6. 'therefore N'   -- conclusion phrases
    """
    # 1
    m = re.search(r"####\s*(-?[\d\.]+)", text)
    if m: return normalise_num(m.group(1))
    # 2
    m = re.search(r"\\boxed\{(-?[\d\.]+)\}", text)
    if m: return normalise_num(m.group(1))
    # 3
    m = re.search(r"(?:the answer is|answer is)\s*:?\s*\$?(-?[\d\.]+)", text, re.IGNORECASE)
    if m: return normalise_num(m.group(1))
    # 4
    m = re.search(r"=\s*\$?(-?[\d\.]+)\s*$", text.strip(), re.MULTILINE)
    if m: return normalise_num(m.group(1))
    # 5
    m = re.search(r"\*\*\$?(-?[\d\.]+)\*\*\.?\s*$", text.strip())
    if m: return normalise_num(m.group(1))
    # 6
    m = re.search(r"(?:therefore|thus|so|hence)[,\s]+(?:the answer is\s*)?\$?(-?[\d\.]+)", text, re.IGNORECASE)
    if m: return normalise_num(m.group(1))
    return ""


# --- Self-test ---
_tests = [
    ("#### 42",             "42"),
    ("#### 3.5",            "3.5"),
    ("\\boxed{100}",        "100"),
    ("The answer is 7",     "7"),
    ("Total = 20",          "20"),
    ("**200**.",            "200"),
    ("Therefore, 13",       "13"),
    ("Some unrelated text", ""),
]
ok = True
for txt, exp in _tests:
    got = extract_pred_answer(txt)
    status = "OK" if got == exp else "FAIL"
    if got != exp: ok = False
    print(f"  {status}  '{txt[:35]}' -> '{got}' (expected '{exp}')")
print("\nAll extractor tests passed" if ok else "\nEXTRACTOR HAS FAILURES -- fix before running eval")


  OK  '#### 42' -> '42' (expected '42')
  OK  '#### 3.5' -> '3.5' (expected '3.5')
  OK  '\boxed{100}' -> '100' (expected '100')
  OK  'The answer is 7' -> '7' (expected '7')
  OK  'Total = 20' -> '20' (expected '20')
  OK  '**200**.' -> '200' (expected '200')
  OK  'Therefore, 13' -> '13' (expected '13')
  OK  'Some unrelated text' -> '' (expected '')

All extractor tests passed


In [7]:
# CELL 7 -- Load fine-tuned guide model (Qwen 3B + LoRA)

def find_adapter():
    patterns = [
        "/kaggle/input/datasets/makkisakib1/final-adapter-qwen-svamp",
        "/kaggle/input/datasets/sufiantabdullah/final-adapter",
        "/kaggle/input/*/adapter",
        "/kaggle/input/*/final-adapter",
        "/kaggle/input/*/final_adapter",
    ]
    for p in patterns:
        for m in glob.glob(p):
            print(f"  Found adapter: {m}")
            return m
    return None


print(f"Loading guide base: {CONFIG['guide_base']}")
guide_tok = AutoTokenizer.from_pretrained(CONFIG["guide_base"], trust_remote_code=True)
guide_tok.padding_side = "left"
if guide_tok.pad_token is None:
    guide_tok.pad_token = guide_tok.eos_token

guide_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["guide_base"],
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

adapter_path = find_adapter()
if adapter_path:
    guide_model = PeftModel.from_pretrained(guide_model, adapter_path)
    print("LoRA adapter loaded -- fine-tuned guide active")
else:
    print("WARNING: No adapter found. Using base Qwen 3B as guide.")

guide_model.eval()
print(f"Guide VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


Loading guide base: Qwen/Qwen2.5-3B-Instruct


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

  Found adapter: /kaggle/input/datasets/makkisakib1/final-adapter-qwen-svamp
LoRA adapter loaded -- fine-tuned guide active
Guide VRAM: 3.14 GB


In [8]:
# CELL 8 -- Load solver model (Qwen 1.5B)

print(f"Loading solver: {CONFIG['response_model']}")
resp_tok = AutoTokenizer.from_pretrained(CONFIG["response_model"])
if resp_tok.pad_token is None:
    resp_tok.pad_token = resp_tok.eos_token

resp_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["response_model"],
    dtype=torch.float16,
    device_map="auto",
).eval()

total_vram = torch.cuda.memory_allocated() / 1e9
headroom   = 17.1 - total_vram
print(f"Total VRAM (both models): {total_vram:.2f} GB / 17.1 GB")
print(f"Headroom                : {headroom:.1f} GB")
print("Memory OK" if headroom >= 2 else "WARNING: Tight -- reduce n_votes to 3 if OOM")


Loading solver: Qwen/Qwen2.5-1.5B-Instruct


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Total VRAM (both models): 4.83 GB / 17.1 GB
Headroom                : 12.3 GB
Memory OK


In [9]:
# CELL 9 -- Prompts and generation functions

GUIDE_SYSTEM = (
    "You are a math problem decomposition assistant.\n"
    "Identify ONLY the arithmetic operations needed. 1-3 steps maximum.\n"
    "State WHAT is being compared or combined using EXACT numbers.\n"
    "Do NOT invent steps. Do NOT reorder the question.\n\n"
    "BAD:  Step 1: Calculate remaining cookies.\n"
    "GOOD: Step 1: Eaten - Given = 14 - 13 = ?\n"
    "      Step 2: Answer = 14 - 13\n\n"
    "If the question asks 'how many MORE did X than Y', "
    "the operation is X - Y, not Y - X.\n"
    "No final answer number. Just the operation steps with actual numbers."
)

SOLVE_SYSTEM = (
    "You are a math problem solver.\n"
    "Compute each step numerically. No markdown. No bullet points. No headers.\n"
    "Write plain arithmetic steps only.\n"
    "Your absolute last line must be: #### [number]\n"
    "NEVER write ### or ** in your response.\n\n"
    "Example:\n"
    "Eaten = 14. Given = 13.\n"
    "Difference = 14 - 13 = 1.\n"
    "#### 1"
)

BASELINE_SYSTEM = (
    "You are a precise math problem solver.\n"
    "Read the problem carefully. Solve step by step, showing every calculation.\n"
    "Your FINAL line must be exactly: #### [number]"
)

REFINER_SYSTEM = (
    "You are a careful math problem solver.\n"
    "Previous attempts on this problem gave different answers.\n"
    "Ignore all previous attempts. Re-solve completely from scratch.\n"
    "Show every arithmetic step.\n"
    "Your FINAL line must be exactly: #### [number]"
)


def run_qwen(mdl, tok, messages, max_tokens, temperature):
    """Call any Qwen-family model and return generated text."""
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=1024)
    device = next(mdl.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = mdl.generate(
            **inputs,
            max_new_tokens     = max_tokens,
            temperature        = max(temperature, 0.05),
            do_sample          = True,
            top_p              = 0.92,
            top_k              = 40,
            pad_token_id       = tok.eos_token_id,
            repetition_penalty = 1.15,
        )
    new_toks = out[0][inputs["input_ids"].shape[1]:]
    return tok.decode(new_toks, skip_special_tokens=True).strip()


def generate_plan(question):
    return run_qwen(
        guide_model, guide_tok,
        [{"role": "system", "content": GUIDE_SYSTEM},
         {"role": "user",   "content": f"Problem: {question}"}],
        max_tokens  = 350,
        temperature = CONFIG["guide_temperature"],
    )


def generate_guided(question, plan):
    content = f"Problem: {question}\n\nPlan (follow each step):\n{plan}\n\nSolve step by step:"
    return run_qwen(
        resp_model, resp_tok,
        [{"role": "system", "content": SOLVE_SYSTEM},
         {"role": "user",   "content": content}],
        max_tokens  = CONFIG["max_new_tokens"],
        temperature = CONFIG["vote_temperature"],
    )


def generate_baseline(question):
    return run_qwen(
        resp_model, resp_tok,
        [{"role": "system", "content": BASELINE_SYSTEM},
         {"role": "user",   "content": f"Problem: {question}"}],
        max_tokens  = CONFIG["max_new_tokens"],
        temperature = CONFIG["vote_temperature"],
    )


def generate_refiner(question, candidates):
    # Refiner runs WITHOUT the plan -- plan may have caused the tie
    cands = ", ".join(sorted(set(c for c in candidates if c)))
    content = (
        f"Problem: {question}\n\n"
        f"Previous attempts gave: {cands}\n"
        "Ignore all previous attempts. Solve from scratch:"
    )
    return run_qwen(
        resp_model, resp_tok,
        [{"role": "system", "content": REFINER_SYSTEM},
         {"role": "user",   "content": content}],
        max_tokens  = CONFIG["max_new_tokens"],
        temperature = CONFIG["refiner_temperature"],
    )


print("Generation functions ready")
print("  Refiner runs WITHOUT plan (avoids inheriting bad plan)")


Generation functions ready
  Refiner runs WITHOUT plan (avoids inheriting bad plan)


In [10]:
# CELL 10 -- Voting logic with richer metrics

def vote_and_decide(answers, question, gt_answer=None):
    """
    Majority voting with refiner fallback on ties.
    Filters empty responses before counting.

    Returns dict with all metrics needed for the three angles.
    """
    valid = [a for a in answers if a and a.strip()]
    if not valid:
        valid = answers  # fallback -- keep all if extraction completely failed

    vote_counts = Counter(valid)
    most_common = vote_counts.most_common()
    top_answer  = most_common[0][0]
    top_count   = most_common[0][1]
    total       = len(valid)

    correct_votes    = vote_counts.get(gt_answer, 0) if gt_answer else 0
    vote_consistency = correct_votes / max(total, 1)
    is_majority      = (len(most_common) == 1 or top_count > most_common[1][1])

    refiner_used    = False
    refiner_correct = None

    if is_majority:
        final    = top_answer
        strategy = "majority"
        conf     = round(top_count / total, 4)
        wasted   = total - top_count
    else:
        # Tie: refiner runs WITHOUT plan
        ref_raw  = generate_refiner(question, list(valid))
        ref_ans  = extract_pred_answer(ref_raw)
        refiner_used    = True
        refiner_correct = (ref_ans == gt_answer) if gt_answer else None

        all_v      = valid + ([ref_ans] if ref_ans else [])
        new_counts = Counter(all_v)
        new_common = new_counts.most_common()
        new_top    = new_common[0][0]
        new_top_c  = new_common[0][1]
        still_tied = len(new_common) > 1 and new_top_c == new_common[1][1]

        final      = new_top
        strategy   = "coin_flip" if still_tied else "refiner_tiebreak"
        conf       = round(new_top_c / len(all_v), 4)
        total      = len(all_v)
        correct_votes    = Counter(all_v).get(gt_answer, 0) if gt_answer else 0
        vote_consistency = correct_votes / max(total, 1)
        wasted           = total - new_top_c
        vote_counts      = new_counts

    return {
        "final_answer"     : final,
        "strategy"         : strategy,
        "confidence"       : conf,
        "vote_counts"      : dict(vote_counts),
        "correct_votes"    : correct_votes,
        "total_votes"      : total,
        "vote_consistency" : round(vote_consistency, 4),
        "wasted_votes"     : wasted,
        "refiner_used"     : refiner_used,
        "refiner_correct"  : refiner_correct,
    }


print("Voting logic ready")
print("  majority         -> clear winner across votes")
print("  refiner_tiebreak -> tie broken by refiner (no plan)")
print("  coin_flip        -> still tied after refiner")


Voting logic ready
  majority         -> clear winner across votes
  refiner_tiebreak -> tie broken by refiner (no plan)
  coin_flip        -> still tied after refiner


In [11]:
# CELL 11 -- Single question test (verify pipeline end-to-end)

print("=" * 65)
print("SINGLE QUESTION TEST  (ASDiv)")
print("=" * 65)

item = test_data[0]
q    = item["question"]
gt   = extract_gt_answer(item["answer"])
op   = item["op_type"]
print(f"Question : {q}")
print(f"GT Answer: {gt}  |  Operation type: {op}")

# Guided
print("\n[1] Guide generating plan...")
plan = generate_plan(q)
print(f"Plan:\n{plan}")

print(f"\n[2] Guided votes ({CONFIG['n_votes']}x)...")
guided_votes = []
for i in range(CONFIG["n_votes"]):
    raw  = generate_guided(q, plan)
    pred = extract_pred_answer(raw)
    guided_votes.append(pred)
    print(f"  Vote {i+1}: '{pred}'  |  raw[:80]: {raw[:80]}")

g = vote_and_decide(guided_votes, q, gt)
print(f"\n  Result    : {g['final_answer']}  (GT: {gt})  {'CORRECT' if g['final_answer']==gt else 'WRONG'}")
print(f"  Strategy  : {g['strategy']}")
print(f"  Confidence: {g['confidence']}")
print(f"  Correct votes: {g['correct_votes']}/{g['total_votes']} ({g['vote_consistency']*100:.0f}%)")

# Baseline
print("\n[3] Baseline votes (no plan)...")
base_votes = []
for i in range(CONFIG["n_votes"]):
    raw  = generate_baseline(q)
    pred = extract_pred_answer(raw)
    base_votes.append(pred)
    print(f"  Vote {i+1}: '{pred}'")

b = vote_and_decide(base_votes, q, gt)
print(f"\n  Baseline result : {b['final_answer']}  (GT: {gt})  {'CORRECT' if b['final_answer']==gt else 'WRONG'}")
print("\nPipeline verified -- run Cell 12 for full evaluation")


SINGLE QUESTION TEST  (ASDiv)
Question : He also has a section filled with short story booklets. If each booklet has 9 pages and there are 49 booklets in the short story section, how many pages will Jack need to go through if he plans to read them all?
GT Answer: 441  |  Operation type: Multiplication

[1] Guide generating plan...
Plan:
Step 1: Booklet Pages * Number of Booklets = 9 * 49 = ?

[2] Guided votes (5x)...
  Vote 1: '441'  |  raw[:80]: Booklet Pages * Number of Booklets

= 9 * 49

= 441
  Vote 2: ''  |  raw[:80]: Step 2: Total Pages Calculation

Step 3: Final Answer

[Total number of pages]
  Vote 3: '441'  |  raw[:80]: Booklet Pages * Number of Booklets  
= 9 * 49  

Multiply 9 by 49:

  = 441

Jac
  Vote 4: ''  |  raw[:80]: Booklet Pages * Number of Booklets = 
9 * 49 = 

Now let's multiply these number
  Vote 5: '441'  |  raw[:80]: ### Step-by-Step Calculation:

1. Multiply the number of pages per booklet by th

  Result    : 441  (GT: 441)  CORRECT
  Strategy  : majority

In [12]:
# CELL 12 -- Full Dual Evaluation Loop
#
# Runs every question TWICE with the SAME questions (seed fixed in Cell 5):
#   Mode A: Guided  (guide plan + solver x5)
#   Mode B: Baseline (solver x5, no plan)
#
# op_type is stored in every record so Angle 4 can group by operation.

print(f"Dual evaluation: {len(test_data)} ASDiv questions")
print(f"Each question: {CONFIG['n_votes']} guided votes + {CONFIG['n_votes']} baseline votes")
print("-" * 65)

all_results  = []
base_results = []
start_idx    = 0

if os.path.exists(CONFIG["checkpoint_file"]):
    with open(CONFIG["checkpoint_file"]) as f:
        ckpt = json.load(f)
    start_idx = ckpt.get("last_index", 0)
    if os.path.exists(CONFIG["results_file"]):
        with open(CONFIG["results_file"]) as f:
            lines = [json.loads(l) for l in f if l.strip()]
        all_results  = [r for r in lines if r.get("mode") == "guided"]
        base_results = [r for r in lines if r.get("mode") == "baseline"]
    print(f"Resumed from index {start_idx}")
    print(f"  Guided saved: {len(all_results)}  Baseline saved: {len(base_results)}")
else:
    print("Starting fresh")

t0 = time.time()

for idx in tqdm(range(start_idx, len(test_data)), desc="ASDiv Eval"):
    item      = test_data[idx]
    question  = item["question"]
    gt_answer = extract_gt_answer(item["answer"])
    op_type   = item["op_type"]

    # ---- GUIDED -----------------------------------------------
    try:
        plan        = generate_plan(question)
        g_votes_raw = [extract_pred_answer(generate_guided(question, plan))
                       for _ in range(CONFIG["n_votes"])]
        g_dec       = vote_and_decide(g_votes_raw, question, gt_answer)

        all_results.append({
            "mode"             : "guided",
            "idx"              : idx,
            "question"         : question,
            "gt_answer"        : gt_answer,
            "op_type"          : op_type,
            "final_answer"     : g_dec["final_answer"],
            "correct"          : g_dec["final_answer"] == gt_answer,
            "strategy"         : g_dec["strategy"],
            "confidence"       : g_dec["confidence"],
            "correct_votes"    : g_dec["correct_votes"],
            "total_votes"      : g_dec["total_votes"],
            "vote_consistency" : g_dec["vote_consistency"],
            "wasted_votes"     : g_dec["wasted_votes"],
            "refiner_used"     : g_dec["refiner_used"],
            "refiner_correct"  : g_dec["refiner_correct"],
            "vote_counts"      : g_dec["vote_counts"],
            "plan"             : plan,
        })
    except RuntimeError as e:
        all_results.append({
            "mode": "guided", "idx": idx, "question": question,
            "gt_answer": gt_answer, "op_type": op_type,
            "final_answer": "", "correct": False,
            "strategy": "error", "confidence": 0.0,
            "correct_votes": 0, "total_votes": CONFIG["n_votes"],
            "vote_consistency": 0.0, "wasted_votes": CONFIG["n_votes"],
            "refiner_used": False, "refiner_correct": None,
            "vote_counts": {}, "error": str(e),
        })

    # ---- BASELINE ---------------------------------------------
    try:
        b_votes_raw = [extract_pred_answer(generate_baseline(question))
                       for _ in range(CONFIG["n_votes"])]
        b_dec       = vote_and_decide(b_votes_raw, question, gt_answer)

        base_results.append({
            "mode"             : "baseline",
            "idx"              : idx,
            "question"         : question,
            "gt_answer"        : gt_answer,
            "op_type"          : op_type,
            "final_answer"     : b_dec["final_answer"],
            "correct"          : b_dec["final_answer"] == gt_answer,
            "strategy"         : b_dec["strategy"],
            "confidence"       : b_dec["confidence"],
            "correct_votes"    : b_dec["correct_votes"],
            "total_votes"      : b_dec["total_votes"],
            "vote_consistency" : b_dec["vote_consistency"],
            "wasted_votes"     : b_dec["wasted_votes"],
            "refiner_used"     : b_dec["refiner_used"],
            "refiner_correct"  : None,
            "vote_counts"      : b_dec["vote_counts"],
        })
    except RuntimeError as e:
        base_results.append({
            "mode": "baseline", "idx": idx, "question": question,
            "gt_answer": gt_answer, "op_type": op_type,
            "final_answer": "", "correct": False,
            "strategy": "error", "confidence": 0.0,
            "correct_votes": 0, "total_votes": CONFIG["n_votes"],
            "vote_consistency": 0.0, "wasted_votes": CONFIG["n_votes"],
            "refiner_used": False, "refiner_correct": None,
            "vote_counts": {}, "error": str(e),
        })

    if (idx + 1) % CONFIG["save_every"] == 0:
        with open(CONFIG["results_file"], "w") as f:
            for r in all_results + base_results:
                f.write(json.dumps(r) + "\n")
        with open(CONFIG["checkpoint_file"], "w") as f:
            json.dump({"last_index": idx + 1}, f)
        g_acc = sum(r["correct"] for r in all_results)  / len(all_results)  * 100
        b_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100
        mins  = (time.time() - t0) / 60
        print(f"  [{idx+1:3d}] Guided: {g_acc:.1f}%  Baseline: {b_acc:.1f}%  ({mins:.1f} min)")

with open(CONFIG["results_file"], "w") as f:
    for r in all_results + base_results:
        f.write(json.dumps(r) + "\n")

g_c = sum(r["correct"] for r in all_results)
b_c = sum(r["correct"] for r in base_results)
print(f"\nEvaluation complete.")
print(f"  Guided   : {g_c}/{len(all_results)} = {g_c/len(all_results)*100:.1f}%")
print(f"  Baseline : {b_c}/{len(base_results)} = {b_c/len(base_results)*100:.1f}%")
print(f"  Delta    : +{(g_c/len(all_results) - b_c/len(base_results))*100:.1f} percentage points")  


Dual evaluation: 300 ASDiv questions
Each question: 5 guided votes + 5 baseline votes
-----------------------------------------------------------------
Starting fresh


ASDiv Eval:   0%|          | 0/300 [00:00<?, ?it/s]

  [ 25] Guided: 60.0%  Baseline: 52.0%  (29.3 min)
  [ 50] Guided: 56.0%  Baseline: 44.0%  (63.8 min)
  [ 75] Guided: 52.0%  Baseline: 45.3%  (96.9 min)
  [100] Guided: 53.0%  Baseline: 46.0%  (128.7 min)
  [125] Guided: 55.2%  Baseline: 48.0%  (157.5 min)
  [150] Guided: 56.7%  Baseline: 48.0%  (188.8 min)
  [175] Guided: 56.6%  Baseline: 47.4%  (216.6 min)
  [200] Guided: 58.5%  Baseline: 48.5%  (247.2 min)
  [225] Guided: 59.6%  Baseline: 51.6%  (271.0 min)
  [250] Guided: 60.4%  Baseline: 52.0%  (299.6 min)
  [275] Guided: 58.9%  Baseline: 51.6%  (337.6 min)
  [300] Guided: 58.7%  Baseline: 49.7%  (375.6 min)

Evaluation complete.
  Guided   : 176/300 = 58.7%
  Baseline : 149/300 = 49.7%
  Delta    : +9.0 percentage points


In [13]:
# CELL 13 -- ANGLE 1: COMPUTE EFFICIENCY
G, S, N = CONFIG["guide_params_B"], CONFIG["solver_params_B"], CONFIG["n_votes"]

guided_compute   = (G * 1) + (S * N)
baseline_compute = S * N
upper_compute    = G * N

g_acc = sum(r["correct"] for r in all_results)  / len(all_results)  * 100
b_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100

g_eff       = g_acc / guided_compute 
b_eff       = b_acc / baseline_compute
savings_pct = (1 - guided_compute / upper_compute) * 100

g_wasted       = sum(r["wasted_votes"] for r in all_results)
b_wasted       = sum(r["wasted_votes"] for r in base_results)
total_possible = len(all_results) * N

ref_triggered = sum(r["refiner_used"] for r in all_results)
ref_correct   = sum(1 for r in all_results if r["refiner_used"] and r.get("refiner_correct"))

strategy_stats = {}
for r in all_results:
    s = r["strategy"]
    if s not in strategy_stats: strategy_stats[s] = {"n":0,"correct":0}
    strategy_stats[s]["n"] += 1
    if r["correct"]: strategy_stats[s]["correct"] += 1

print("=" * 65)
print("ANGLE 1 -- COMPUTE EFFICIENCY  (ASDiv)")
print("=" * 65)
print(f"\n  {'Setup':<32} | {'Compute':>10} | {'Accuracy':>9} | {'Acc/B':>7}")
print(f"  {'-'*32}-+-{'-'*10}-+-{'-'*9}-+-{'-'*7}")
print(f"  {'Baseline (1.5B x ' + str(N) + ')':<32} | {baseline_compute:>8.1f}B  | {b_acc:>8.1f}% | {b_eff:>6.3f}")
print(f"  {'Guided  (3B x1 + 1.5B x' + str(N) + ')':<32} | {guided_compute:>8.1f}B  | {g_acc:>8.1f}% | {g_eff:>6.3f}")
print(f"  {'Upper   (3B x ' + str(N) + ')':<32} | {upper_compute:>8.1f}B  | {'(ceiling)':>9} |")
print(f"\n  Accuracy gain over baseline  : +{g_acc - b_acc:.1f} percentage points")
print(f"  Compute savings vs upper     : {savings_pct:.0f}% cheaper")
print(f"  Wasted votes saved           : {b_wasted - g_wasted}  ({g_wasted} guided vs {b_wasted} baseline)")
if ref_triggered:
    print(f"  Refiner: {ref_triggered} triggered, {ref_correct} correct ({ref_correct/ref_triggered*100:.1f}%)")
print(f"\n  Strategy breakdown (guided):")
for s, v in sorted(strategy_stats.items(), key=lambda x: -x[1]["n"]):
    acc_s = v["correct"]/v["n"]*100 if v["n"] else 0
    print(f"    {s:<22}: {v['n']:>4} questions  {acc_s:>6.1f}% accuracy")

angle1 = {
    "dataset": "ASDiv", "n_questions": len(all_results),
    "guided_compute_B": guided_compute, "baseline_compute_B": baseline_compute,
    "upper_compute_B": upper_compute, "guided_accuracy": round(g_acc,2),
    "baseline_accuracy": round(b_acc,2), "accuracy_gain": round(g_acc-b_acc,2),
    "compute_savings_pct": round(savings_pct,1),
    "guided_efficiency": round(g_eff,4), "baseline_efficiency": round(b_eff,4),
    "guided_wasted_votes": g_wasted, "baseline_wasted_votes": b_wasted,
    "wasted_votes_saved": b_wasted-g_wasted,
    "refiner_triggered": ref_triggered, "refiner_correct": ref_correct,
    "strategy_breakdown": strategy_stats,
}
with open(CONFIG["angle1_file"], "w") as f:
    json.dump(angle1, f, indent=2)
print(f"\nSaved -> {CONFIG['angle1_file']}")


ANGLE 1 -- COMPUTE EFFICIENCY  (ASDiv)

  Setup                            |    Compute |  Accuracy |   Acc/B
  ---------------------------------+------------+-----------+--------
  Baseline (1.5B x 5)              |      7.5B  |     49.7% |  6.622
  Guided  (3B x1 + 1.5B x5)        |     10.5B  |     58.7% |  5.587
  Upper   (3B x 5)                 |     15.0B  | (ceiling) |

  Accuracy gain over baseline  : +9.0 percentage points
  Compute savings vs upper     : 30% cheaper
  Wasted votes saved           : 11  (290 guided vs 301 baseline)
  Refiner: 44 triggered, 13 correct (29.5%)

  Strategy breakdown (guided):
    majority              :  256 questions    62.5% accuracy
    coin_flip             :   29 questions    27.6% accuracy
    refiner_tiebreak      :   15 questions    53.3% accuracy

Saved -> /kaggle/working/asdiv_eval/angle1_compute_efficiency.json


In [14]:
# CELL 14 -- ANGLE 2: VOTE CONSISTENCY
g_cons = [r["vote_consistency"] for r in all_results]
b_cons = [r["vote_consistency"] for r in base_results]

g_mean = np.mean(g_cons)
b_mean = np.mean(b_cons)
lift   = g_mean / max(b_mean, 1e-6)

guided_wins   = sum(1 for g, b in zip(g_cons, b_cons) if g > b)
baseline_wins = sum(1 for g, b in zip(g_cons, b_cons) if b > g)
tied          = sum(1 for g, b in zip(g_cons, b_cons) if g == b)

def bucket(scores):
    return {
        "all_wrong  (0%)":  sum(1 for s in scores if s == 0.0),
        "low       (1-39%)":sum(1 for s in scores if 0.0 < s < 0.4),
        "medium  (40-79%)": sum(1 for s in scores if 0.4 <= s < 0.8),
        "high   (80-100%)": sum(1 for s in scores if s >= 0.8),
    }

g_dist = bucket(g_cons)
b_dist = bucket(b_cons)

g_corr = [r["vote_consistency"] for r in all_results  if r["correct"]]
b_corr = [r["vote_consistency"] for r in base_results if r["correct"]]

print("=" * 65)
print("ANGLE 2 -- VOTE CONSISTENCY  (ASDiv)")
print("=" * 65)
print(f"\n  Mean correct-vote ratio (out of {CONFIG['n_votes']} votes per question):")
print(f"    Guided   : {g_mean*100:.1f}%  ({g_mean*CONFIG['n_votes']:.2f} votes correct on average)")
print(f"    Baseline : {b_mean*100:.1f}%  ({b_mean*CONFIG['n_votes']:.2f} votes correct on average)")
print(f"    Lift     : {lift:.2f}x  (guided produces {lift:.1f}x more correct votes per question)")
print(f"\n  Per-question comparison (same questions, both modes):")
print(f"    Guided beats baseline : {guided_wins} / {len(all_results)} questions")
print(f"    Baseline beats guided : {baseline_wins} / {len(all_results)} questions")
print(f"    Equal                 : {tied} / {len(all_results)} questions")
print(f"\n  {'Bucket':<22} | {'Guided':>8} | {'Baseline':>8} | {'Diff':>6}")
print(f"  {'-'*22}-+-{'-'*8}-+-{'-'*8}-+-{'-'*6}")
for bkt in ["all_wrong  (0%)", "low       (1-39%)", "medium  (40-79%)", "high   (80-100%)"]:
    gv, bv = g_dist[bkt], b_dist[bkt]
    sign = "+" if gv-bv >= 0 else ""
    print(f"  {bkt:<22} | {gv:>8} | {bv:>8} | {sign+str(gv-bv):>6}")
if g_corr:
    print(f"\n  Correct-question consistency: Guided={np.mean(g_corr)*100:.1f}%  Baseline={np.mean(b_corr)*100:.1f}%")
    print("    (High consistency + correct = genuine reliable solving, not lucky vote)")

angle2 = {
    "dataset": "ASDiv", "n_questions": len(all_results),
    "guided_mean_consistency": round(g_mean,4), "baseline_mean_consistency": round(b_mean,4),
    "consistency_lift": round(lift,4), "guided_wins": guided_wins,
    "baseline_wins": baseline_wins, "tied": tied,
    "guided_distribution": g_dist, "baseline_distribution": b_dist,
    "guided_correct_q_consistency":   round(np.mean(g_corr),4) if g_corr else 0,
    "baseline_correct_q_consistency": round(np.mean(b_corr),4) if b_corr else 0,
}
with open(CONFIG["angle2_file"], "w") as f:
    json.dump(angle2, f, indent=2)
print(f"\nSaved -> {CONFIG['angle2_file']}")


ANGLE 2 -- VOTE CONSISTENCY  (ASDiv)

  Mean correct-vote ratio (out of 5 votes per question):
    Guided   : 52.6%  (2.63 votes correct on average)
    Baseline : 45.4%  (2.27 votes correct on average)
    Lift     : 1.16x  (guided produces 1.2x more correct votes per question)

  Per-question comparison (same questions, both modes):
    Guided beats baseline : 123 / 300 questions
    Baseline beats guided : 82 / 300 questions
    Equal                 : 95 / 300 questions

  Bucket                 |   Guided | Baseline |   Diff
  -----------------------+----------+----------+-------
  all_wrong  (0%)        |       83 |       90 |     -7
  low       (1-39%)      |       43 |       48 |     -5
  medium  (40-79%)       |       62 |       88 |    -26
  high   (80-100%)       |      112 |       74 |    +38

  Correct-question consistency: Guided=83.2%  Baseline=78.5%
    (High consistency + correct = genuine reliable solving, not lucky vote)

Saved -> /kaggle/working/asdiv_eval/angle2_vo

In [15]:
# CELL 15 -- ANGLE 3: CONFIDENCE CALIBRATION
def calibration_report(results, label):
    buckets = [
        ("Very High  (>=0.80)", lambda c: c >= 0.80, 0.90),
        ("High       (0.60-0.80)", lambda c: 0.60 <= c < 0.80, 0.70),
        ("Medium     (0.40-0.60)", lambda c: 0.40 <= c < 0.60, 0.50),
        ("Low        (<0.40)",  lambda c: c < 0.40, 0.25),
    ]
    n_total    = len(results)
    ece        = 0.0
    calib_out  = []
    false_conf = sum(1 for r in results if r["confidence"] >= 0.80 and not r["correct"])

    print(f"\n  [{label}]")
    print(f"  {'Confidence':<26} | {'N':>5} | {'Accuracy':>9} | {'Expected':>9} | {'Gap':>6} | Cal?")
    print(f"  {'-'*26}-+-{'-'*5}-+-{'-'*9}-+-{'-'*9}-+-{'-'*6}-+----")
    for name, cond, mid in buckets:
        subset = [r for r in results if cond(r["confidence"])]
        if not subset:
            print(f"  {name:<26} | {'--':>5} | {'--':>9} | {mid*100:>8.0f}% | {'--':>6} |")
            continue
        n   = len(subset)
        acc = sum(r["correct"] for r in subset) / n
        gap = abs(acc - mid)
        ece += (n / n_total) * gap
        flag = "Good" if gap < 0.15 else "Poor"
        print(f"  {name:<26} | {n:>5} | {acc*100:>8.1f}% | {mid*100:>8.0f}% | {gap:>6.3f} | {flag}")
        calib_out.append({"bucket":name,"count":n,"accuracy":round(acc,4),
                           "expected":mid,"gap":round(gap,4)})
    hc     = [r for r in results if r["confidence"] >= 0.80]
    hc_acc = sum(r["correct"] for r in hc) / max(1, len(hc)) * 100
    print(f"  {'ECE':<26}   {ece:.4f}")
    print(f"  High-conf: {len(hc)} questions  |  Accuracy: {hc_acc:.1f}%  |  Confidently WRONG: {false_conf}")
    return ece, calib_out, false_conf


print("=" * 65)
print("ANGLE 3 -- CONFIDENCE CALIBRATION  (ASDiv)")
print("=" * 65)
print("Ideal: accuracy at each confidence level matches that level.")
print("False confidence: model agrees on wrong answer with full certainty.")

g_ece, g_calib, g_false = calibration_report(all_results,  "GUIDED pipeline")
b_ece, b_calib, b_false = calibration_report(base_results, "BASELINE (no plan)")

improve = (b_ece - g_ece) / max(b_ece, 1e-6) * 100
print(f"\n  ECE Summary:")
print(f"    Guided   ECE : {g_ece:.4f}")
print(f"    Baseline ECE : {b_ece:.4f}")
print(f"    Improvement  : {improve:.1f}% better calibrated")
print(f"\n  False Confidence: Guided={g_false}  Baseline={b_false}  Reduction={b_false-g_false}")

angle3 = {
    "dataset": "ASDiv", "n_questions": len(all_results),
    "guided_ece": round(g_ece,4), "baseline_ece": round(b_ece,4),
    "ece_improvement_pct": round(improve,2),
    "guided_false_confidence": g_false, "baseline_false_confidence": b_false,
    "false_conf_reduction": b_false-g_false,
    "guided_calibration": g_calib, "baseline_calibration": b_calib,
}
with open(CONFIG["angle3_file"], "w") as f:
    json.dump(angle3, f, indent=2)
print(f"\nSaved -> {CONFIG['angle3_file']}")


ANGLE 3 -- CONFIDENCE CALIBRATION  (ASDiv)
Ideal: accuracy at each confidence level matches that level.
False confidence: model agrees on wrong answer with full certainty.

  [GUIDED pipeline]
  Confidence                 |     N |  Accuracy |  Expected |    Gap | Cal?
  ---------------------------+-------+-----------+-----------+--------+----
  Very High  (>=0.80)        |   158 |     70.9% |       90% |  0.191 | Poor
  High       (0.60-0.80)     |    81 |     50.6% |       70% |  0.194 | Poor
  Medium     (0.40-0.60)     |    34 |     44.1% |       50% |  0.059 | Good
  Low        (<0.40)         |    27 |     29.6% |       25% |  0.046 | Good
  ECE                          0.1638
  High-conf: 158 questions  |  Accuracy: 70.9%  |  Confidently WRONG: 46

  [BASELINE (no plan)]
  Confidence                 |     N |  Accuracy |  Expected |    Gap | Cal?
  ---------------------------+-------+-----------+-----------+--------+----
  Very High  (>=0.80)        |   129 |     57.4% |       9

In [16]:
# CELL 16 -- ANGLE 4: ACCURACY BY OPERATION TYPE  (ASDiv exclusive)
# =================================================================
# This is the analysis only ASDiv enables.
# We measure guided vs baseline accuracy and vote consistency
# separately for each operation category:
#   Addition / Subtraction / Multiplication / Division / Multi-step
#
# This answers: "Does guidance help more on harder operation types?"
# Expected finding: guidance lifts more on Multiplication and Division
# than on Addition, because those require more structured reasoning.
# =================================================================

g_by_op = defaultdict(list)
b_by_op = defaultdict(list)
for r in all_results:
    g_by_op[r.get("op_type", "Unknown")].append(r)
for r in base_results:
    b_by_op[r.get("op_type", "Unknown")].append(r)

all_ops = sorted(set(list(g_by_op.keys()) + list(b_by_op.keys())))

print("=" * 72)
print("ANGLE 4 -- ACCURACY BY OPERATION TYPE  (ASDiv exclusive)")
print("=" * 72)
print(f"\n  {'Operation':<16} | {'N':>4} | {'Guided':>8} | {'Baseline':>9} | {'Gain':>6} | {'Consistency lift':>17}")
print(f"  {'-'*16}-+-{'-'*4}-+-{'-'*8}-+-{'-'*9}-+-{'-'*6}-+-{'-'*17}")

op_results = {}
for op in all_ops:
    g_items = g_by_op.get(op, [])
    b_items = b_by_op.get(op, [])
    n = len(g_items)
    if n == 0: continue

    g_acc  = sum(r["correct"] for r in g_items) / n * 100
    b_acc  = sum(r["correct"] for r in b_items) / max(len(b_items),1) * 100
    gain   = g_acc - b_acc
    sign   = "+" if gain >= 0 else ""

    g_cons = np.mean([r["vote_consistency"] for r in g_items]) * 100
    b_cons = np.mean([r["vote_consistency"] for r in b_items]) * 100 if b_items else 0
    c_lift = g_cons / max(b_cons, 1e-6)

    print(f"  {op:<16} | {n:>4} | {g_acc:>7.1f}% | {b_acc:>8.1f}% | {sign+str(round(gain,1))+'%':>6} | {c_lift:>6.2f}x  ({g_cons:.1f}% vs {b_cons:.1f}%)")
    op_results[op] = {
        "n": n,
        "guided_acc": round(g_acc,2), "baseline_acc": round(b_acc,2),
        "gain": round(gain,2),
        "guided_consistency": round(g_cons,2), "baseline_consistency": round(b_cons,2),
        "consistency_lift": round(c_lift,4),
    }

# Summary: which operation type benefits most from guidance?
gains = sorted(op_results.items(), key=lambda x: -x[1]["gain"])
print(f"\n  Operation types ranked by guidance gain:")
for op, v in gains:
    sign = "+" if v["gain"] >= 0 else ""
    print(f"    {op:<16}: {sign}{v['gain']}%  (guided={v['guided_acc']}%  baseline={v['baseline_acc']}%)")

angle4 = {"dataset": "ASDiv", "by_operation_type": op_results}
with open(CONFIG["angle4_file"], "w") as f:
    json.dump(angle4, f, indent=2)
print(f"\nSaved -> {CONFIG['angle4_file']}")


ANGLE 4 -- ACCURACY BY OPERATION TYPE  (ASDiv exclusive)

  Operation        |    N |   Guided |  Baseline |   Gain |  Consistency lift
  -----------------+------+----------+-----------+--------+------------------
  Addition         |   61 |    73.8% |     60.7% | +13.1% |   1.17x  (64.8% vs 55.5%)
  Division         |   40 |    40.0% |     20.0% | +20.0% |   1.83x  (41.0% vs 22.5%)
  Multi-step       |   89 |    46.1% |     38.2% |  +7.9% |   1.11x  (40.7% vs 36.6%)
  Multiplication   |   43 |    69.8% |     69.8% |  +0.0% |   1.06x  (63.4% vs 59.9%)
  Subtraction      |   67 |    65.7% |     59.7% |  +6.0% |   1.09x  (57.2% vs 52.3%)

  Operation types ranked by guidance gain:
    Division        : +20.0%  (guided=40.0%  baseline=20.0%)
    Addition        : +13.11%  (guided=73.77%  baseline=60.66%)
    Multi-step      : +7.87%  (guided=46.07%  baseline=38.2%)
    Subtraction     : +5.97%  (guided=65.67%  baseline=59.7%)
    Multiplication  : +0.0%  (guided=69.77%  baseline=69.77%)



In [17]:
# CELL 17 -- Full Paper Summary (all four angles)

with open(CONFIG["angle1_file"]) as f: a1 = json.load(f)
with open(CONFIG["angle2_file"]) as f: a2 = json.load(f)
with open(CONFIG["angle3_file"]) as f: a3 = json.load(f)
with open(CONFIG["angle4_file"]) as f: a4 = json.load(f)

n = a1["n_questions"]

print("=" * 68)
print("  ASDiv EVALUATION -- PAPER SUMMARY TABLE")
print("=" * 68)
print(f"  Dataset: ASDiv  |  N={n}  |  Seed={CONFIG['random_seed']}")
print(f"  Models : Qwen 2.5-3B guide + Qwen 2.5-1.5B solver")
print()

rows = [
    ["Metric",                   "Baseline",      "Guided",         "Change"],
    ["Overall Accuracy",
     str(a1['baseline_accuracy']) + "%",
     str(a1['guided_accuracy']) + "%",
     "+" + str(round(a1['guided_accuracy']-a1['baseline_accuracy'],1)) + " pts"],
    ["Compute Cost",
     str(a1['baseline_compute_B']) + "B param-passes",
     str(a1['guided_compute_B']) + "B param-passes",
     str(a1['compute_savings_pct']) + "% cheaper than ceiling"],
    ["Wasted Votes",
     str(a1['baseline_wasted_votes']),
     str(a1['guided_wasted_votes']),
     str(a1['wasted_votes_saved']) + " fewer"],
    ["Vote Consistency",
     str(round(a2['baseline_mean_consistency']*100,1)) + "%",
     str(round(a2['guided_mean_consistency']*100,1)) + "%",
     str(round(a2['consistency_lift'],2)) + "x lift"],
    ["High-Agreement Questions",
     str(a2['baseline_distribution']['high   (80-100%)']),
     str(a2['guided_distribution']['high   (80-100%)']),
     ""],
    ["Guided Wins Per-Question",
     "--",
     str(a2['guided_wins']) + " / " + str(n),
     ""],
    ["ECE (lower = better)",
     str(a3['baseline_ece']),
     str(a3['guided_ece']),
     str(a3['ece_improvement_pct']) + "% better"],
    ["False Confidence Count",
     str(a3['baseline_false_confidence']),
     str(a3['guided_false_confidence']),
     str(a3['false_conf_reduction']) + " fewer"],
]

col_w = [28, 20, 20, 28]
sep   = "-+-".join("-" * w for w in col_w)
for i, row in enumerate(rows):
    line = " | ".join(str(cell).ljust(col_w[j]) for j, cell in enumerate(row))
    print("  " + line)
    if i == 0:
        print("  " + sep)

if a1.get("refiner_triggered", 0) > 0:
    rt = a1["refiner_triggered"]
    rc = a1.get("refiner_correct", 0)
    print(f"\n  Refiner: triggered {rt} times, resolved {rc} correctly ({rc/rt*100:.1f}%)")

# Angle 4 summary
print(f"\n  Accuracy gain by operation type (Angle 4):")
gains = sorted(a4["by_operation_type"].items(), key=lambda x: -x[1]["gain"])
for op, v in gains:
    sign = "+" if v["gain"] >= 0 else ""
    print(f"    {op:<16}: {sign}{v['gain']}%  "
          f"(guided={v['guided_acc']}%  baseline={v['baseline_acc']}%  "
          f"n={v['n']})")

full = {
    "dataset": "ASDiv", "seed": CONFIG["random_seed"],
    "n_questions": n, "angle1": a1, "angle2": a2, "angle3": a3, "angle4": a4,
}
with open(CONFIG["report_file"], "w") as f:
    json.dump(full, f, indent=2)

print(f"\nAll results saved to {OUTPUT_DIR}")
print("Commit this notebook to preserve outputs.")


  ASDiv EVALUATION -- PAPER SUMMARY TABLE
  Dataset: ASDiv  |  N=300  |  Seed=42
  Models : Qwen 2.5-3B guide + Qwen 2.5-1.5B solver

  Metric                       | Baseline             | Guided               | Change                      
  -----------------------------+----------------------+----------------------+-----------------------------
  Overall Accuracy             | 49.67%               | 58.67%               | +9.0 pts                    
  Compute Cost                 | 7.5B param-passes    | 10.5B param-passes   | 30.0% cheaper than ceiling  
  Wasted Votes                 | 301                  | 290                  | 11 fewer                    
  Vote Consistency             | 45.4%                | 52.6%                | 1.16x lift                  
  High-Agreement Questions     | 74                   | 112                  |                             
  Guided Wins Per-Question     | --                   | 123 / 300            |                             
  